# COMET Enrich Diffs

Per enrichment method, stats for the old and new releases and what changed between them.

In [1]:
import warnings

# jupysql's parse.py has invalid escape sequences that warn on Python 3.12+.
warnings.filterwarnings("ignore", category=SyntaxWarning)

%reload_ext sql
%sql duckdb:///:memory:
%config SqlMagic.displaylimit = 0
%config SqlMagic.autopandas = False

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'duckdb:///:memory:'

## Paths

Each path points at a release folder, the folder containing `manifest.json` and `enrichments/`.

In [2]:
datasets = {
    "resource-type-general": {
        "old": "../../comet-enrich/data/diff-smoke/resource-type-general/old",
        "new": "../../comet-enrich/data/diff-smoke/resource-type-general/new",
        "diff": "../../comet-enrich/data/diff-smoke/resource-type-general/diff",
    },
    "funders": {
        "old": "../../comet-enrich/data/diff-smoke/funders/old",
        "new": "../../comet-enrich/data/diff-smoke/funders/new",
        "diff": "../../comet-enrich/data/diff-smoke/funders/diff",
    },
    "affiliations": {
        "old": "../../comet-enrich/data/diff-smoke/affiliations/old",
        "new": "../../comet-enrich/data/diff-smoke/affiliations/new",
        "diff": "../../comet-enrich/data/diff-smoke/affiliations/diff",
    },
}

## Releases

- **enrichment records**: the release manifest's `report.counters.emitted`.
- **DOIs**: distinct DOIs with at least one enrichment. Lower than records wherever a DOI has several enrichments.

In [3]:
%%sql

{% for method, paths in datasets.items() %}
{% set first_method = loop.first %}
{% for release in ["old", "new"] %}
{% if not (first_method and loop.first) %}UNION ALL{% endif %}
SELECT
    '{{ method }}' AS method,
    '{{ release }}' AS release,
    manifest.sources.datacite.release_date AS "DataCite date",
    format('{:,}', manifest.report.counters.emitted) AS "enrichment records",
    format('{:,}', (
        SELECT COUNT(DISTINCT doi)
        FROM read_json_auto('{{ paths[release] }}/enrichments/*.jsonl.gz')
    )) AS "DOIs"
FROM read_json_auto('{{ paths[release] }}/manifest.json') AS manifest
{% endfor %}
{% endfor %}

Running query in 'duckdb:///:memory:'

method,release,DataCite date,enrichment records,DOIs
resource-type-general,old,2026-08-03,"4,054,747","4,054,747"
resource-type-general,new,2026-09-03,"4,059,930","4,059,930"
funders,old,2026-08-03,"1,165,588","881,518"
funders,new,2026-09-03,"1,173,321","883,601"
affiliations,old,2026-08-03,"25,334,300","6,893,868"
affiliations,new,2026-09-03,"27,345,405","7,194,338"


## Diff

The records columns are the diff manifest counters and count enrichment records, not DOIs.

- **DOIs that lost all enrichments**: DOIs with a retraction in the diff that have no enrichments in the new release.
- **DOIs with enrichments for the first time**: DOIs with an assertion in the diff that had no enrichments in the old release.

For each method, DOIs in the new release minus DOIs in the old release equals DOIs with enrichments for the first time minus DOIs that lost all enrichments.

In [4]:
%%sql

{% for method, paths in datasets.items() %}
{% if not loop.first %}UNION ALL{% endif %}
SELECT
    '{{ method }}' AS method,
    format('{:,}', counters.asserted) AS "asserted records",
    format('{:,}', counters.retracted) AS "retracted records",
    format('{:,}', counters.superseded) AS "superseded records",
    format('{:,}', (
        SELECT COUNT(*)
        FROM (
            SELECT doi
            FROM read_json_auto('{{ paths.diff }}/enrichments/*.jsonl.gz')
            WHERE event = 'retracted'
            EXCEPT
            SELECT doi
            FROM read_json_auto('{{ paths.new }}/enrichments/*.jsonl.gz')
        ) t
    )) AS "DOIs that lost all enrichments",
    format('{:,}', (
        SELECT COUNT(*)
        FROM (
            SELECT doi
            FROM read_json_auto('{{ paths.diff }}/enrichments/*.jsonl.gz')
            WHERE event = 'asserted'
            EXCEPT
            SELECT doi
            FROM read_json_auto('{{ paths.old }}/enrichments/*.jsonl.gz')
        ) t
    )) AS "DOIs with enrichments for the first time"
FROM read_json_auto('{{ paths.diff }}/manifest.json')
{% endfor %}

Running query in 'duckdb:///:memory:'

method,asserted records,retracted records,superseded records,DOIs that lost all enrichments,DOIs with enrichments for the first time
resource-type-general,"24,933","19,750",0,"19,611","24,794"
funders,"10,436","2,703",0,"1,866","3,949"
affiliations,"2,061,235","50,130",0,"11,581","312,051"
